# 03 — RFQ Flow Analysis

Analyze the simulated RFQ flow:
- Intraday patterns (should peak around midday)
- Month-end spikes
- Client-bond pair coverage (sparsity structure)
- Program trade signatures
- MMPP regime effects on flow


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rfq_sim.utils.io import load_all

data = load_all('../data')
obs  = data['observable']
obs['timestamp'] = pd.to_datetime(obs['timestamp'])
obs['hour']   = obs['timestamp'].dt.hour
obs['date']   = obs['timestamp'].dt.date
obs['month']  = obs['timestamp'].dt.month
obs['dayofweek'] = obs['timestamp'].dt.dayofweek

print(f"Total RFQs: {len(obs):,}")
print(f"Date range: {obs['timestamp'].min().date()} to {obs['timestamp'].max().date()}")


In [ ]:
# Intraday flow pattern
# Should show the sin^2 shape with peak around 11:30-13:00

hourly = obs.groupby('hour').size()
fig, ax = plt.subplots(figsize=(10, 4))
hourly.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Hour of Day (ET)')
ax.set_ylabel('RFQ Count')
ax.set_title('Intraday RFQ Flow (should peak ~midday, thin at open/close)')
ax.axvline(x=hourly.idxmax() - 7, color='red', linestyle='--', alpha=0.5,
           label=f'Peak at {hourly.idxmax()}:00')
ax.legend()
plt.tight_layout()
plt.savefig('../data/plots_intraday_flow.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Daily flow -- look for month-end spikes
daily_counts = obs.groupby('date').size().reset_index(name='count')
daily_counts['date'] = pd.to_datetime(daily_counts['date'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(daily_counts['date'], daily_counts['count'],
       color='steelblue', alpha=0.7, width=0.8)

# Mark month-end days (last 2 business days per month)
ax.set_xlabel('Date')
ax.set_ylabel('Daily RFQ Count')
ax.set_title('Daily RFQ Flow (month-end spikes visible as bumps at month boundaries)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/plots_daily_flow.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Client-bond pair sparsity
# With affinity structure, most pairs should be zero but high-affinity
# pairs should have significantly more interactions

K = obs['client_id'].nunique()
N = obs['bond_id'].nunique()
pair_counts = obs.groupby(['client_id', 'bond_id']).size()

total_possible = K * N
observed_pairs  = len(pair_counts)
sparsity = 1.0 - observed_pairs / total_possible

print(f"Clients: {K}, Bonds: {N}")
print(f"Possible pairs: {total_possible:,}")
print(f"Observed pairs: {observed_pairs:,}")
print(f"Sparsity: {sparsity:.3%}")
print(f"Mean RFQs per observed pair: {pair_counts.mean():.2f}")

# Distribution of pair interaction counts
fig, ax = plt.subplots(figsize=(8, 4))
pair_counts.value_counts().sort_index().head(20).plot(kind='bar', ax=ax)
ax.set_xlabel('Number of RFQs per (client, bond) pair')
ax.set_ylabel('Number of pairs')
ax.set_title('RFQ count distribution per client-bond pair (long tail expected)')
plt.tight_layout()
plt.show()


In [ ]:
# Program trade signatures: clients in program state should show
# elevated activity on correlated bonds in short windows

prog_rfqs = obs[obs['client_in_program'] == True]
non_prog  = obs[obs['client_in_program'] == False]

print(f"Program state RFQs: {len(prog_rfqs):,} ({len(prog_rfqs)/len(obs):.1%})")

# In program state: compare sector concentration vs non-program
if len(prog_rfqs) > 10 and len(non_prog) > 10:
    prog_sector_entropy = (
        prog_rfqs.groupby('client_id')['bond_sector']
        .apply(lambda x: -(x.value_counts(normalize=True) *
                           np.log(x.value_counts(normalize=True) + 1e-10)).sum())
    ).mean()

    non_prog_sector_entropy = (
        non_prog.groupby('client_id')['bond_sector']
        .apply(lambda x: -(x.value_counts(normalize=True) *
                           np.log(x.value_counts(normalize=True) + 1e-10)).sum())
    ).mean()

    print(f"Mean sector entropy (program):     {prog_sector_entropy:.3f}")
    print(f"Mean sector entropy (non-program): {non_prog_sector_entropy:.3f}")
    print("Lower entropy in program state = more sector concentration = good signal")
